# MobileUNETR Training for Image Segmentation

This notebook trains the MobileUNETR model for image segmentation using a custom dataset structure:
- train/images (.jpg)
- train/masks (.png binary images with values 0 and 255)
- val/images (.jpg)
- val/masks (.png binary images with values 0 and 255)
- test/images (.jpg - for submission to Kaggle)

In [ ]:
import os
import sys
import torch
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from glob import glob
from tqdm.auto import tqdm

# Add the project root to the path to import MobileUNETR modules
sys.path.append(os.getcwd())

# Import MobileUNETR modules
from architectures.mobileunetr import build_mobileunetr_s, build_mobileunetr_xs, build_mobileunetr_xxs
from augmentations.segmentation_augmentations import build_augmentations
from losses.losses import build_loss_fn
from optimizers.optimizers import build_optimizer
from optimizers.schedulers import build_scheduler
from train_scripts.utils import seed_everything, save_checkpoint, log_metrics

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seed for reproducibility
SEED = 42
seed_everything(SEED)

## 1. Data Preparation

First, we'll create custom datasets for loading and augmenting images and masks.

In [ ]:
# Define data directories - update these to your dataset paths
BASE_PATH = "path/to/your/dataset"  # Change this to your dataset path
TRAIN_IMAGES = os.path.join(BASE_PATH, "train/images")
TRAIN_MASKS = os.path.join(BASE_PATH, "train/masks")
VAL_IMAGES = os.path.join(BASE_PATH, "val/images")
VAL_MASKS = os.path.join(BASE_PATH, "val/masks") 
TEST_IMAGES = os.path.join(BASE_PATH, "test/images")

# Create output directory for test predictions
OUTPUT_DIR = os.path.join(BASE_PATH, "predictions")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify dataset structure
print(f"Train images: {len(os.listdir(TRAIN_IMAGES))}")
print(f"Train masks: {len(os.listdir(TRAIN_MASKS))}")
print(f"Validation images: {len(os.listdir(VAL_IMAGES))}")
print(f"Validation masks: {len(os.listdir(VAL_MASKS))}")
print(f"Test images: {len(os.listdir(TEST_IMAGES))}")

## 2. Custom Dataset

We'll create a custom dataset class to handle our image and mask data.

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, images_dir, masks_dir=None, transforms=None, test_mode=False):
        """
        Custom dataset for segmentation tasks
        
        Args:
            images_dir (str): Directory containing images
            masks_dir (str, optional): Directory containing masks
            transforms: Albumentations transforms
            test_mode (bool): If True, doesn't load masks (for test set)
        """
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transforms = transforms
        self.test_mode = test_mode
        
        # Get all image files
        self.image_files = sorted(glob(os.path.join(images_dir, "*.jpg")))
        
        if not self.test_mode and masks_dir is not None:
            # Ensure mask files have the same base names as image files
            self.mask_files = []
            for img_path in self.image_files:
                img_filename = os.path.basename(img_path)
                mask_filename = os.path.splitext(img_filename)[0] + ".png"
                mask_path = os.path.join(masks_dir, mask_filename)
                if os.path.exists(mask_path):
                    self.mask_files.append(mask_path)
                else:
                    print(f"Warning: Mask not found for {img_path}")
                    self.mask_files.append(None)
        
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        # Load image
        img_path = self.image_files[idx]
        image = np.array(Image.open(img_path).convert("RGB"), dtype=np.uint8)
        
        # For test mode, only return the image
        if self.test_mode:
            if self.transforms:
                transformed = self.transforms(image=image)
                image = transformed["image"]
            
            # Return image and filename for identification during prediction
            return {
                "image": image, 
                "filename": os.path.basename(img_path)
            }
        
        # Load mask for training/validation
        mask_path = self.mask_files[idx]
        mask = np.array(Image.open(mask_path), dtype=np.uint8)
        
        # Normalize mask to 0 and 1 (from 0 and 255)
        mask = mask // 255
        
        # Apply transformations
        if self.transforms:
            transformed = self.transforms(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]
        
        # Return as dictionary
        return {
            "image": image,
            "mask": mask.unsqueeze(0)  # Add channel dimension [1, H, W]
        }

## 3. Define Augmentations and Dataloaders

In [ ]:
# Configuration for image size, augmentations
IMAGE_SIZE = (512, 512)  # Common size for segmentation tasks
BATCH_SIZE = 8
NUM_WORKERS = 4

# Define mean and std for normalization
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Create augmentations
def get_transforms(phase):
    augmentation_args = {
        "mean": MEAN,
        "std": STD,
        "image_size": IMAGE_SIZE
    }
    
    if phase == "train":
        return build_augmentations(train=True, augmentation_args=augmentation_args)
    else:
        return build_augmentations(train=False, augmentation_args=augmentation_args)

# Create datasets
train_dataset = SegmentationDataset(
    images_dir=TRAIN_IMAGES,
    masks_dir=TRAIN_MASKS,
    transforms=get_transforms("train")
)

val_dataset = SegmentationDataset(
    images_dir=VAL_IMAGES,
    masks_dir=VAL_MASKS,
    transforms=get_transforms("val")
)

test_dataset = SegmentationDataset(
    images_dir=TEST_IMAGES,
    masks_dir=None,
    transforms=get_transforms("val"),
    test_mode=True
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# Verify the datasets
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

## 4. Visualize Some Training Examples

In [ ]:
def visualize_sample(dataset, idx=0):
    """Visualize a sample image and its mask"""
    sample = dataset[idx]
    
    # If using test dataset, only visualize the image
    if isinstance(sample, dict) and "filename" in sample:
        plt.figure(figsize=(10, 5))
        
        # Convert tensor to numpy if needed
        image = sample["image"]
        if torch.is_tensor(image):
            image = image.permute(1, 2, 0).numpy()
            # Denormalize
            image = image * np.array(STD) + np.array(MEAN)
            image = np.clip(image, 0, 1)
        
        plt.imshow(image)
        plt.title(f"Test Image: {sample['filename']}")
        plt.axis("off")
        plt.show()
        return
    
    # For train/val datasets with mask
    image = sample["image"]
    mask = sample["mask"]
    
    # Convert tensors to numpy if needed
    if torch.is_tensor(image):
        image = image.permute(1, 2, 0).numpy()
        # Denormalize
        image = image * np.array(STD) + np.array(MEAN)
        image = np.clip(image, 0, 1)
    
    if torch.is_tensor(mask):
        mask = mask.squeeze().numpy()
    
    # Display image and mask
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 3, 1)
    plt.imshow(image)
    plt.title("Image")
    plt.axis("off")
    
    plt.subplot(1, 3, 2)
    plt.imshow(mask, cmap='gray')
    plt.title("Mask")
    plt.axis("off")
    
    plt.subplot(1, 3, 3)
    plt.imshow(image)
    plt.imshow(mask, alpha=0.4, cmap='jet')
    plt.title("Overlay")
    plt.axis("off")
    
    plt.tight_layout()
    plt.show()

# Visualize a few training samples
for i in range(3):
    idx = random.randint(0, len(train_dataset) - 1)
    print(f"Sample {i+1}, Index: {idx}")
    visualize_sample(train_dataset, idx)

## 5. Build Model and Training Configuration

In [ ]:
# Model configuration
NUM_CLASSES = 1  # Binary segmentation
MODEL_TYPE = "mobileunetr_xxs"  # Options: mobileunetr_s, mobileunetr_xs, mobileunetr_xxs

# Build model
model = None
if MODEL_TYPE == "mobileunetr_s":
    model = build_mobileunetr_s(num_classes=NUM_CLASSES, image_size=IMAGE_SIZE)
elif MODEL_TYPE == "mobileunetr_xs":
    model = build_mobileunetr_xs(num_classes=NUM_CLASSES, image_size=IMAGE_SIZE)
elif MODEL_TYPE == "mobileunetr_xxs":
    model = build_mobileunetr_xxs(num_classes=NUM_CLASSES, image_size=IMAGE_SIZE)
else:
    raise ValueError(f"Unknown model type: {MODEL_TYPE}")

# Move model to device
model = model.to(device)

# Get the number of trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {MODEL_TYPE}")
print(f"Number of trainable parameters: {trainable_params:,}")

# Training configuration
NUM_EPOCHS = 50
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5
LOSS_TYPE = "dice"  # Options: dice, dicebce, binarycrossentropy

# Build loss function
criterion = build_loss_fn(loss_type=LOSS_TYPE)

# Build optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=5,
    verbose=True,
    min_lr=1e-6
)

# Create directory for saving checkpoints
CHECKPOINT_DIR = os.path.join(BASE_PATH, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Training configuration:")
print(f"- Number of epochs: {NUM_EPOCHS}")
print(f"- Learning rate: {LEARNING_RATE}")
print(f"- Loss function: {LOSS_TYPE}")
print(f"- Batch size: {BATCH_SIZE}")
print(f"- Image size: {IMAGE_SIZE}")

## 6. Define Evaluation Metrics

In [ ]:
# Define evaluation metrics
def dice_coefficient(y_pred, y_true, smooth=1.0):
    """Calculate Dice coefficient between predicted and ground truth masks"""
    y_pred = torch.sigmoid(y_pred).flatten()
    y_true = y_true.flatten()
    intersection = (y_pred * y_true).sum()
    return (2.0 * intersection + smooth) / (y_pred.sum() + y_true.sum() + smooth)

def iou_score(y_pred, y_true, smooth=1.0):
    """Calculate IoU score between predicted and ground truth masks"""
    y_pred = torch.sigmoid(y_pred).flatten()
    y_true = y_true.flatten()
    intersection = (y_pred * y_true).sum()
    union = y_pred.sum() + y_true.sum() - intersection
    return (intersection + smooth) / (union + smooth)

## 7. Training and Validation Functions

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Train the model for one epoch"""
    model.train()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    
    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    
    for batch_idx, data in enumerate(progress_bar):
        # Get data
        images = data["image"].to(device)
        masks = data["mask"].to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        
        # Calculate loss
        loss = criterion(outputs, masks)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        # Calculate metrics
        with torch.no_grad():
            batch_dice = dice_coefficient(outputs, masks).item()
            batch_iou = iou_score(outputs, masks).item()
        
        # Update running metrics
        running_loss += loss.item()
        running_dice += batch_dice
        running_iou += batch_iou
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'dice': f'{batch_dice:.4f}',
            'iou': f'{batch_iou:.4f}'
        })
    
    # Calculate epoch metrics
    epoch_loss = running_loss / len(dataloader)
    epoch_dice = running_dice / len(dataloader)
    epoch_iou = running_iou / len(dataloader)
    
    return epoch_loss, epoch_dice, epoch_iou


def validate(model, dataloader, criterion, device):
    """Validate the model on validation set"""
    model.eval()
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    
    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc="Validation", leave=False)
        
        for batch_idx, data in enumerate(progress_bar):
            # Get data
            images = data["image"].to(device)
            masks = data["mask"].to(device)
            
            # Forward pass
            outputs = model(images)
            
            # Calculate loss
            loss = criterion(outputs, masks)
            
            # Calculate metrics
            batch_dice = dice_coefficient(outputs, masks).item()
            batch_iou = iou_score(outputs, masks).item()
            
            # Update running metrics
            running_loss += loss.item()
            running_dice += batch_dice
            running_iou += batch_iou
            
            # Update progress bar
            progress_bar.set_postfix({
                'val_loss': f'{loss.item():.4f}',
                'val_dice': f'{batch_dice:.4f}',
                'val_iou': f'{batch_iou:.4f}'
            })
    
    # Calculate epoch metrics
    epoch_loss = running_loss / len(dataloader)
    epoch_dice = running_dice / len(dataloader)
    epoch_iou = running_iou / len(dataloader)
    
    return epoch_loss, epoch_dice, epoch_iou

## 8. Training Loop

In [ ]:
# Training history
history = {
    'train_loss': [],
    'train_dice': [],
    'train_iou': [],
    'val_loss': [],
    'val_dice': [],
    'val_iou': [],
}

# Track best model
best_dice = 0.0
best_model_path = os.path.join(CHECKPOINT_DIR, f"{MODEL_TYPE}_best_model.pth")

# Start training
for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    
    # Train for one epoch
    train_loss, train_dice, train_iou = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    # Validate
    val_loss, val_dice, val_iou = validate(
        model, val_loader, criterion, device
    )
    
    # Update learning rate based on validation metric
    scheduler.step(val_dice)
    
    # Record history
    history['train_loss'].append(train_loss)
    history['train_dice'].append(train_dice)
    history['train_iou'].append(train_iou)
    history['val_loss'].append(val_loss)
    history['val_dice'].append(val_dice)
    history['val_iou'].append(val_iou)
    
    # Print epoch metrics
    print(f"Train Loss: {train_loss:.4f}, Dice: {train_dice:.4f}, IoU: {train_iou:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Dice: {val_dice:.4f}, IoU: {val_iou:.4f}")
    
    # Save best model
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': val_loss,
            'dice': val_dice,
            'iou': val_iou,
        }, best_model_path)
        print(f"Saved best model with dice: {best_dice:.4f}")

# Save final model
final_model_path = os.path.join(CHECKPOINT_DIR, f"{MODEL_TYPE}_final_model.pth")
torch.save({
    'epoch': NUM_EPOCHS,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': val_loss,
    'dice': val_dice,
    'iou': val_iou,
}, final_model_path)

print(f"Training complete. Best validation Dice: {best_dice:.4f}")

## 9. Plot Training History

In [ ]:
# Plot training history
plt.figure(figsize=(15, 5))

# Plot loss
plt.subplot(1, 3, 1)
plt.plot(history['train_loss'], label='Train')
plt.plot(history['val_loss'], label='Validation')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot Dice
plt.subplot(1, 3, 2)
plt.plot(history['train_dice'], label='Train')
plt.plot(history['val_dice'], label='Validation')
plt.title('Dice Coefficient')
plt.xlabel('Epoch')
plt.ylabel('Dice')
plt.legend()
plt.grid(True)

# Plot IoU
plt.subplot(1, 3, 3)
plt.plot(history['train_iou'], label='Train')
plt.plot(history['val_iou'], label='Validation')
plt.title('IoU Score')
plt.xlabel('Epoch')
plt.ylabel('IoU')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'training_history.png'))
plt.show()

## 10. Predict on Test Data and Save Results

In [ ]:
# Load the best model for inference
def load_best_model(model, model_path):
    """Load the best model for inference"""
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model, checkpoint

# Load best model for inference
model, checkpoint = load_best_model(model, best_model_path)
print(f"Loaded best model from epoch {checkpoint['epoch']+1} with dice: {checkpoint['dice']:.4f}")

# Function to predict and save masks
def predict_and_save_masks(model, dataloader, output_dir):
    """Predict masks for test images and save them"""
    model.eval()
    os.makedirs(output_dir, exist_ok=True)
    
    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc="Predicting", total=len(dataloader))
        
        for batch in progress_bar:
            # Get images and filenames
            images = batch["image"].to(device)
            filenames = batch["filename"]
            
            # Forward pass
            outputs = model(images)
            preds = torch.sigmoid(outputs)
            
            # Convert predictions to binary masks and save
            preds = (preds > 0.5).float() * 255
            preds = preds.cpu().numpy().astype(np.uint8)
            
            # Save each mask in the batch
            for i, filename in enumerate(filenames):
                # Create output filename
                base_name = os.path.splitext(filename)[0]
                output_path = os.path.join(output_dir, f"{base_name}.png")
                
                # Save the mask as PNG
                mask = preds[i, 0]  # Get the mask (remove batch and channel dims)
                Image.fromarray(mask).save(output_path)
                
    print(f"Saved predictions to {output_dir}")

# Predict on test data and save masks
predict_and_save_masks(model, test_loader, OUTPUT_DIR)

# Visualize some predictions
def visualize_predictions(test_dir, pred_dir, num_samples=3):
    """Visualize some test images and their predicted masks"""
    test_images = sorted(glob(os.path.join(test_dir, "*.jpg")))
    
    # Select random samples
    if len(test_images) > num_samples:
        indices = random.sample(range(len(test_images)), num_samples)
        test_images = [test_images[i] for i in indices]
    
    for img_path in test_images:
        # Load image
        image = np.array(Image.open(img_path).convert("RGB"))
        
        # Load prediction
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        pred_path = os.path.join(pred_dir, f"{base_name}.png")
        
        if os.path.exists(pred_path):
            pred_mask = np.array(Image.open(pred_path))
            
            # Display image and prediction
            plt.figure(figsize=(12, 5))
            
            plt.subplot(1, 3, 1)
            plt.imshow(image)
            plt.title("Original Image")
            plt.axis("off")
            
            plt.subplot(1, 3, 2)
            plt.imshow(pred_mask, cmap='gray')
            plt.title("Predicted Mask")
            plt.axis("off")
            
            plt.subplot(1, 3, 3)
            plt.imshow(image)
            plt.imshow(pred_mask, alpha=0.4, cmap='jet')
            plt.title("Overlay")
            plt.axis("off")
            
            plt.suptitle(f"Prediction for {base_name}")
            plt.tight_layout()
            plt.show()
        else:
            print(f"Prediction not found for {base_name}")

# Visualize some predictions
visualize_predictions(TEST_IMAGES, OUTPUT_DIR)

## 11. Summary

The MobileUNETR model has been successfully:
1. Trained on the custom segmentation dataset
2. Validated for performance
3. Used to generate predictions on the test set
4. The predictions have been saved to the specified output directory

The best model has been saved at: `{CHECKPOINT_DIR}/{MODEL_TYPE}_best_model.pth`